# Notebook 13 — Supervisor Implementation & Validation

Loads the best model from Notebook 12 and wraps it in a supervisory controller.
Structure:
1. Setup + helpers (reuse scenario generation from NB12)
2. Phase 1 — initial parameter sweeps (original supervisor)
3. Phase 2 — bug discovery, corrected supervisor
4. Three-strategy comparison
5. Full 10-patient validation
6. Final 3-config comparison + report figures

In [ ]:
# config
from datetime import datetime
from enum import Enum

MODEL_ROOT = "outputs_v14_final_dataset"
OUT_ROOT   = "outputs_v15_final_supervisor"

START      = datetime(2024, 1, 1)
WINDOW_MIN = 90

TEST_PATIENT = "adult#002"
SEED_START   = 12000
MAX_TRIES    = 500

# case severity bands (mg/dL)
MILD_LO, MILD_HI = 55, 70
SEVERE_HI = 54

N_REPEATS = 20   # sensor-noise repeats for stats

In [ ]:
import os, random, copy
from pathlib import Path
from datetime import datetime, timedelta
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import joblib

try:
    from scipy.stats import wilcoxon
    HAS_SCIPY = True
except ImportError:
    HAS_SCIPY = False

try:
    from IPython.display import display
except ImportError:
    def display(x): print(x)

from simglucose.simulation.env import T1DSimEnv
from simglucose.patient.t1dpatient import T1DPatient
from simglucose.sensor.cgm import CGMSensor
from simglucose.actuator.pump import InsulinPump
from simglucose.simulation.scenario import CustomScenario
from simglucose.controller.basal_bolus_ctrller import BBController
from simglucose.controller.base import Action

OUT  = Path(OUT_ROOT)
FIGS = OUT / "figures"
CSVS = OUT / "tables"
for d in [OUT, FIGS, CSVS]:
    d.mkdir(parents=True, exist_ok=True)

In [ ]:
# load trained model from NB12
MDIR = Path(MODEL_ROOT) / "models"
clf          = joblib.load(MDIR / "best_model.joblib")
feature_cols = joblib.load(MDIR / "best_feature_cols.joblib")
model_label  = joblib.load(MDIR / "best_model_label.joblib")

IS_EXTENDED = len(feature_cols) > 10
print(f"Loaded: {model_label} ({len(feature_cols)} features, extended={IS_EXTENDED})")

In [ ]:
# reproducibility + skill helper
def hard_reset(seed=123):
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    random.seed(seed)

def get_skill(patient):
    idx = int(patient.split("#")[1]) - 1
    return ["good", "avg", "poor"][idx % 3]

In [ ]:
# scenario helpers — must be identical to NB12 for same-seed reproducibility
def _clip(x, lo, hi): return float(max(lo, min(hi, x)))

def partial_eating(schedule, frac_by_time):
    return [(t, float(g) * float(frac_by_time.get(t, 1.0))) for t, g in schedule]

def delay_meals(schedule, delay_by_time):
    out = [(t + timedelta(minutes=int(delay_by_time.get(t, 0))), float(g)) for t, g in schedule]
    return sorted(out, key=lambda x: x[0])

def add_snacks(schedule, snacks):
    return sorted(list(schedule) + [(t, float(g)) for t, g in snacks], key=lambda x: x[0])

def split_meal(schedule, meal_time, frac_now=0.6, delay_min=90):
    out, found = [], False
    for t, g in schedule:
        if t == meal_time and not found:
            found = True
            g = float(g)
            out.append((t, frac_now * g))
            out.append((t + timedelta(minutes=int(delay_min)), (1 - frac_now) * g))
        else:
            out.append((t, float(g)))
    return sorted(out, key=lambda x: x[0])

def generate_day_scenario(start, rng, base_meals=None, patient_skill="avg"):
    if base_meals is None:
        base_meals = [
            (start.replace(hour=8, minute=0, second=0, microsecond=0), 60.0),
            (start.replace(hour=13, minute=0, second=0, microsecond=0), 80.0),
            (start.replace(hour=19, minute=0, second=0, microsecond=0), 70.0),
        ]
    if patient_skill == "good":    sd_b, sd_l, sd_d, p_bl = 0.07, 0.12, 0.15, 0.03
    elif patient_skill == "poor":  sd_b, sd_l, sd_d, p_bl = 0.12, 0.22, 0.28, 0.08
    else:                          sd_b, sd_l, sd_d, p_bl = 0.09, 0.16, 0.20, 0.06

    u = rng.random()
    if u < 0.40:   dt = "normal"
    elif u < 0.60: dt = "rushed"
    elif u < 0.78: dt = "under_eat"
    elif u < 0.92: dt = "takeaway_dinner"
    else:          dt = "blunder"
    if dt == "rushed": sd_l *= 1.3; sd_d *= 1.3

    sch = [(t, float(g)) for t, g in base_meals]
    delays = {}
    for t, _ in sch:
        if dt in ("rushed", "takeaway_dinner"):
            delays[t] = int(_clip(rng.normal(15, 15), 0, 45))
        else:
            delays[t] = int(_clip(rng.normal(0, 10), -15, 15))
    sch = delay_meals(sch, delays)
    ms = sorted(sch, key=lambda x: x[0])
    t_bk, t_ln, t_dn = ms[0][0], ms[1][0], ms[2][0]

    ue, ue_tgt, ue_f = False, None, 1.0
    if dt == "under_eat":
        ue = True; ue_tgt = rng.choice(["lunch", "dinner"])
        ue_f = float(_clip(rng.normal(0.75, 0.10), 0.55, 0.90))
        sch = partial_eating(sch, {t_ln if ue_tgt == "lunch" else t_dn: ue_f})

    sp, sp_f, sp_d = False, None, None
    if dt == "takeaway_dinner":
        sp = True
        sp_f = float(_clip(rng.normal(0.6, 0.08), 0.45, 0.75))
        sp_d = int(_clip(rng.normal(90, 20), 60, 140))
        sch = split_meal(sch, t_dn, frac_now=sp_f, delay_min=sp_d)

    snks = []
    if dt in ("rushed", "takeaway_dinner", "blunder"):
        for _ in range(int(rng.choice([0, 1, 2], p=[0.45, 0.40, 0.15]))):
            hr = int(rng.choice([15, 16, 17, 20, 21], p=[0.18, 0.22, 0.22, 0.20, 0.18]))
            mn = int(rng.choice([0, 10, 20, 30, 40, 50]))
            snks.append((start.replace(hour=hr, minute=mn, second=0, microsecond=0),
                         float(_clip(rng.normal(18, 6), 8, 35))))
    sch = add_snacks(sch, snks)

    def sample_k(sd):
        k = float(_clip(rng.normal(1.0, sd), 0.6, 1.6))
        if rng.random() < p_bl or dt == "blunder":
            k = float(rng.uniform(1.6, 2.4)) if rng.random() < 0.7 else float(rng.uniform(0.35, 0.65))
        return k

    kb, kl, kd = sample_k(sd_b), sample_k(sd_l), sample_k(sd_d)
    k_map = {t_bk: kb, t_ln: kl, t_dn: kd}
    meta = {"day_type": dt, "patient_skill": patient_skill,
            "delay_breakfast_min": delays.get(base_meals[0][0], 0),
            "delay_lunch_min": delays.get(base_meals[1][0], 0),
            "delay_dinner_min": delays.get(base_meals[2][0], 0),
            "k_breakfast": kb, "k_lunch": kl, "k_dinner": kd,
            "under_eat": ue, "under_eat_target": ue_tgt or "", "under_eat_frac": float(ue_f),
            "split_meal": sp, "split_frac_now": float(sp_f) if sp else 0.0,
            "split_delay_min": int(sp_d) if sp else 0, "n_snacks": len(snks)}
    return sch, k_map, meta

def build_announced_schedule(true_sch, k_map):
    return {t: float(k_map[t]) * float(g) for t, g in true_sch if t in k_map}

In [ ]:
# online feature builder — mirrors NB12 but works on a rolling buffer
def build_online_features(cgm_hist, dt_min):
    s = pd.Series(cgm_hist, dtype=float)
    w15 = max(1, int(round(15 / dt_min)))
    w30 = max(1, int(round(30 / dt_min)))
    w45 = max(1, int(round(45 / dt_min)))
    w60 = max(1, int(round(60 / dt_min)))

    f = {}
    f["CGM"]         = float(s.iloc[-1])
    f["cgm_mean_30"] = float(s.rolling(w30, min_periods=1).mean().iloc[-1])
    f["cgm_std_30"]  = float(s.rolling(w30, min_periods=1).std().fillna(0).iloc[-1])
    f["cgm_min_30"]  = float(s.rolling(w30, min_periods=1).min().iloc[-1])
    f["cgm_max_30"]  = float(s.rolling(w30, min_periods=1).max().iloc[-1])
    f["cgm_delta_15"] = float(s.diff(w15).fillna(0).iloc[-1])
    f["cgm_delta_30"] = float(s.diff(w30).fillna(0).iloc[-1])

    if IS_EXTENDED:
        f["cgm_mean_15"] = float(s.rolling(w15, min_periods=1).mean().iloc[-1])
        f["cgm_std_15"]  = float(s.rolling(w15, min_periods=1).std().fillna(0).iloc[-1])
        f["cgm_min_15"]  = float(s.rolling(w15, min_periods=1).min().iloc[-1])
        f["cgm_max_15"]  = float(s.rolling(w15, min_periods=1).max().iloc[-1])
        f["cgm_mean_45"] = float(s.rolling(w45, min_periods=1).mean().iloc[-1])
        f["cgm_std_45"]  = float(s.rolling(w45, min_periods=1).std().fillna(0).iloc[-1])
        f["cgm_min_45"]  = float(s.rolling(w45, min_periods=1).min().iloc[-1])
        f["cgm_mean_60"] = float(s.rolling(w60, min_periods=1).mean().iloc[-1])
        f["cgm_std_60"]  = float(s.rolling(w60, min_periods=1).std().fillna(0).iloc[-1])
        f["cgm_min_60"]  = float(s.rolling(w60, min_periods=1).min().iloc[-1])
        f["cgm_delta_60"] = float(s.diff(w60).fillna(0).iloc[-1])
        d15 = s.diff(w15).fillna(0)
        f["cgm_accel_15"] = float(d15.diff(w15).fillna(0).iloc[-1])

    X = np.array([[float(f.get(c, 0)) for c in feature_cols]])
    slope = f["cgm_delta_15"] / 15.0
    return X, f, slope

## Supervisor classes

Two versions:
- **Original** — no training-domain guard, raw delta slope gate (used for initial sweeps)
- **Corrected** — adds hard-rule below 80 mg/dL, converts slope to mg/dL/min

In [ ]:
class SupervisorMode(Enum):
    ATTENUATION = "attenuation"
    RESCUE_CARB = "rescue_carb"
    HYBRID      = "hybrid"

@dataclass
class SupervisorConfig:
    mode:          SupervisorMode
    alert_th:      float = 0.95
    cgm_gate:      float = 100.0
    slope_gate:    float = -1.0
    n_consecutive: int   = 1
    hold_min:      int   = 30
    cooldown_min:  int   = 60
    history_min:   int   = 3
    basal_scale:   float = 0.0
    bolus_scale:   float = 0.0
    rescue_carb_g: float = 0.0


class OriginalSupervisor:
    """First version — no domain guard, raw delta for slope gate."""
    def __init__(self, clf, feature_cols, cfg):
        self.clf = clf
        self.feature_cols = list(feature_cols)
        self.cfg = cfg
        self.active_until = None
        self.cooldown_until = None
        self.risk_hits = 0
        self.intervention_delivered = False

    def decide(self, t, cgm_now, dt_min, cgm_hist):
        if len(cgm_hist) < self.cfg.history_min:
            return {"p": np.nan, "active": False, "source": "insufficient_history"}

        X, feats, _ = build_online_features(cgm_hist, dt_min)
        p = self.clf.predict_proba(
            pd.DataFrame([feats])[self.feature_cols].values)[:, 1][0]

        # original: raw delta, no domain guard
        gate_ok = (cgm_now <= self.cfg.cgm_gate) and (feats["cgm_delta_15"] <= self.cfg.slope_gate)

        if self.active_until and t < self.active_until:
            return {"p": p, "active": True, "source": "hold"}
        if self.cooldown_until and t < self.cooldown_until:
            self.risk_hits = 0
            self.intervention_delivered = False
            return {"p": p, "active": False, "source": "cooldown"}

        self.intervention_delivered = False
        if gate_ok and p >= self.cfg.alert_th:
            self.risk_hits += 1
        else:
            self.risk_hits = 0

        active = self.risk_hits >= self.cfg.n_consecutive
        if active:
            self.active_until = t + pd.Timedelta(minutes=self.cfg.hold_min)
            self.cooldown_until = self.active_until + pd.Timedelta(minutes=self.cfg.cooldown_min)
        return {"p": p, "active": active, "source": "ml_no_guard"}

    def get_intervention(self, action_base, active):
        action_mod, rescue = action_base, 0.0
        if not active:
            return {"action": action_mod, "rescue_carbs": rescue}
        if self.cfg.mode in (SupervisorMode.ATTENUATION, SupervisorMode.HYBRID):
            action_mod = Action(
                basal=float(action_base.basal) * self.cfg.basal_scale,
                bolus=float(action_base.bolus) * self.cfg.bolus_scale)
        if self.cfg.mode in (SupervisorMode.RESCUE_CARB, SupervisorMode.HYBRID):
            if not self.intervention_delivered:
                rescue = float(self.cfg.rescue_carb_g)
                self.intervention_delivered = True
        return {"action": action_mod, "rescue_carbs": rescue}


class PairedRiskSupervisor:
    """
    Corrected version with two fixes:
    1) Training-domain guard: if CGM <= 80, bypass model and activate directly
    2) Slope gate uses mg/dL per minute instead of raw 15-min delta
    """
    MODEL_FLOOR = 80.0  # must match obs_cgm_min from NB12

    def __init__(self, clf, feature_cols, cfg):
        self.clf = clf
        self.feature_cols = list(feature_cols)
        self.cfg = cfg
        self.active_until = None
        self.cooldown_until = None
        self.risk_hits = 0
        self.intervention_delivered = False

    def decide(self, t, cgm_now, dt_min, cgm_hist):
        if len(cgm_hist) < self.cfg.history_min:
            return {"p": np.nan, "active": False, "source": "insufficient_history"}

        # cooldown
        if self.cooldown_until and t < self.cooldown_until:
            self.risk_hits = 0
            self.intervention_delivered = False
            p = self._model_prob(cgm_hist, dt_min) if cgm_now > self.MODEL_FLOOR else np.nan
            return {"p": p, "active": False, "source": "cooldown"}

        # hold
        if self.active_until and t < self.active_until:
            p = self._model_prob(cgm_hist, dt_min) if cgm_now > self.MODEL_FLOOR else np.nan
            return {"p": p, "active": True, "source": "hold"}

        self.intervention_delivered = False

        # case 1: below training domain — hard rule
        if cgm_now <= self.MODEL_FLOOR:
            self.risk_hits = self.cfg.n_consecutive
            self._set_timers(t)
            return {"p": np.nan, "active": True, "source": "hard_rule_below_80"}

        # case 2: model is valid
        X, feats, _ = build_online_features(cgm_hist, dt_min)
        p = float(self.clf.predict_proba(
            pd.DataFrame([feats])[self.feature_cols].values)[:, 1][0])

        cgm_ok  = cgm_now <= self.cfg.cgm_gate
        slope   = feats["cgm_delta_15"] / 15.0   # mg/dL per minute
        slope_ok = slope <= self.cfg.slope_gate

        if cgm_ok and slope_ok and p >= self.cfg.alert_th:
            self.risk_hits += 1
        else:
            self.risk_hits = 0

        active = self.risk_hits >= self.cfg.n_consecutive
        if active:
            self._set_timers(t)
        return {"p": p, "active": active, "source": "ml_model"}

    def _set_timers(self, t):
        self.active_until = t + pd.Timedelta(minutes=self.cfg.hold_min)
        self.cooldown_until = self.active_until + pd.Timedelta(minutes=self.cfg.cooldown_min)

    def _model_prob(self, cgm_hist, dt_min):
        try:
            X, feats, _ = build_online_features(cgm_hist, dt_min)
            return float(self.clf.predict_proba(
                pd.DataFrame([feats])[self.feature_cols].values)[:, 1][0])
        except Exception:
            return np.nan

    def get_intervention(self, action_base, active):
        action_mod, rescue = action_base, 0.0
        if not active:
            return {"action": action_mod, "rescue_carbs": rescue}
        if self.cfg.mode in (SupervisorMode.ATTENUATION, SupervisorMode.HYBRID):
            action_mod = Action(
                basal=float(action_base.basal) * self.cfg.basal_scale,
                bolus=float(action_base.bolus) * self.cfg.bolus_scale)
        if self.cfg.mode in (SupervisorMode.RESCUE_CARB, SupervisorMode.HYBRID):
            if not self.intervention_delivered:
                rescue = float(self.cfg.rescue_carb_g)
                self.intervention_delivered = True
        return {"action": action_mod, "rescue_carbs": rescue}

In [ ]:
# 24h paired rollout — runs baseline + supervisor side by side
def run_day(patient_name, true_schedule, announced_schedule,
            sensor_seed=123, supervisor=None):
    hard_reset(sensor_seed)
    true_sch = copy.deepcopy(true_schedule)
    ann_sch  = copy.deepcopy(announced_schedule)

    start = min(t for t, _ in true_sch)
    scenario = CustomScenario(start_time=start, scenario=true_sch)
    patient  = T1DPatient.withName(patient_name)
    sensor   = CGMSensor.withName("Dexcom", seed=int(sensor_seed))
    pump     = InsulinPump.withName("Insulet")
    env      = T1DSimEnv(patient=patient, sensor=sensor, pump=pump, scenario=scenario)
    ctrl     = BBController()
    step     = env.reset()
    dt_min   = float(step.info["sample_time"])
    t_end    = step.info["time"] + timedelta(hours=24)

    rows, cgm_hist = [], []

    while step.info["time"] < t_end:
        t = pd.Timestamp(step.info["time"])
        true_meal = float(step.info["meal"])
        meal_ann  = float(ann_sch.get(t.to_pydatetime(), 0.0))

        action_base = ctrl.policy(
            step.observation, step.reward, step.done,
            meal=meal_ann, patient_name=step.info["patient_name"])

        cgm_now = float(step.observation.CGM)
        cgm_hist.append(cgm_now)
        rescue_now = 0.0
        src = "none"

        if supervisor is None:
            p, active = np.nan, False
            action_mod = action_base
        else:
            dec = supervisor.decide(t=t, cgm_now=cgm_now, dt_min=dt_min, cgm_hist=cgm_hist)
            p = float(dec["p"]) if not np.isnan(dec["p"]) else np.nan
            active = bool(dec["active"])
            src = dec.get("source", "unknown")
            intervention = supervisor.get_intervention(action_base, active)
            action_mod = intervention["action"]
            rescue_now = intervention["rescue_carbs"]
            if rescue_now > 0:
                env.scenario.scenario.append((t.to_pydatetime(), rescue_now))
                env.scenario.scenario.sort(key=lambda x: x[0])

        rows.append({
            "time": t, "CGM": cgm_now, "BG": float(step.info["bg"]),
            "true_meal_signal": true_meal, "announced_meal_signal": meal_ann,
            "p_hypo": p, "supervisor_active": int(active),
            "decision_source": src,
            "basal_base": float(action_base.basal), "bolus_base": float(action_base.bolus),
            "basal_mod": float(action_mod.basal), "bolus_mod": float(action_mod.bolus),
            "rescue_carbs": rescue_now,
        })
        step = env.step(action_mod)

    return pd.DataFrame(rows)

In [ ]:
# metrics + case-finding helpers
def metrics(df):
    cgm = df["CGM"].astype(float).values
    return {
        "minCGM": float(np.min(cgm)),
        "TBR70":  float(np.mean(cgm < 70) * 100),
        "TBR54":  float(np.mean(cgm < 54) * 100),
        "TIR":    float(np.mean((cgm >= 70) & (cgm <= 180)) * 100),
        "TAR":    float(np.mean(cgm > 180) * 100),
        "active_pct": float(df["supervisor_active"].mean() * 100) if "supervisor_active" in df.columns else 0,
        "total_rescue_carbs": float(df["rescue_carbs"].sum()) if "rescue_carbs" in df.columns else 0,
    }

def find_case(patient, target="mild", seed_start=SEED_START, max_tries=MAX_TRIES):
    skill = get_skill(patient)
    for seed in range(seed_start, seed_start + max_tries):
        rng = np.random.default_rng(seed)
        sch, k, meta = generate_day_scenario(START, rng=rng, patient_skill=skill)
        ann = build_announced_schedule(sch, k)
        df = run_day(patient, sch, ann, sensor_seed=seed)
        m = metrics(df)

        if target == "mild" and (MILD_LO <= m["minCGM"] < MILD_HI):
            print(f"  Found mild case: seed={seed}, minCGM={m['minCGM']:.1f}")
            return {"patient": patient, "scenario_seed": seed,
                    "true_schedule": sch, "announced_schedule": ann, "meta": meta}
        elif target == "severe" and m["minCGM"] < SEVERE_HI:
            print(f"  Found severe case: seed={seed}, minCGM={m['minCGM']:.1f}")
            return {"patient": patient, "scenario_seed": seed,
                    "true_schedule": sch, "announced_schedule": ann, "meta": meta}
    print(f"  No {target} case found for {patient}")
    return None

def find_case_range(patient, lo, hi, seed_start=SEED_START, max_tries=MAX_TRIES):
    skill = get_skill(patient)
    for seed in range(seed_start, seed_start + max_tries):
        rng = np.random.default_rng(seed)
        sch, k, meta = generate_day_scenario(START, rng=rng, patient_skill=skill)
        ann = build_announced_schedule(sch, k)
        df = run_day(patient, sch, ann, sensor_seed=seed)
        m = metrics(df)
        if lo <= m["minCGM"] <= hi:
            print(f"  {patient}: seed={seed}, minCGM={m['minCGM']:.1f}")
            return {"patient": patient, "scenario_seed": seed,
                    "true_schedule": sch, "announced_schedule": ann,
                    "baseline_minCGM": m["minCGM"], "meta": meta}
    print(f"  No case for {patient}")
    return None

def safe_wilcoxon(x, y, alt):
    x, y = np.asarray(x, float), np.asarray(y, float)
    if not HAS_SCIPY or len(x) != len(y) or np.allclose(x, y):
        return np.nan if not np.allclose(x, y) else 1.0
    try:
        return float(wilcoxon(x, y, alternative=alt)[1])
    except Exception:
        return np.nan

def shade_active(ax, times, active, alpha=0.12):
    active = np.asarray(active).astype(bool)
    times = pd.to_datetime(times)
    start = None
    for i in range(len(active)):
        if active[i] and start is None:
            start = times.iloc[i]
        if start and (i == len(active)-1 or (active[i] and not active[i+1])):
            ax.axvspan(start, times.iloc[i], alpha=alpha, color="orange")
            start = None

## Phase 1 — Parameter Sweeps (Original Supervisor)

Sweep each parameter individually to map the operating space.
Using the original supervisor here — issues found during these sweeps
motivated the corrections in Phase 2.

In [ ]:
# use original supervisor for phase 1
Supervisor = OriginalSupervisor

case_ph1 = find_case(TEST_PATIENT, target="mild")
assert case_ph1 is not None

def run_sweep(param_name, values, make_cfg):
    """Generic sweep runner — avoids repeating the same loop 5 times."""
    rows = []
    for val in values:
        print(f"  {param_name}={val}")
        for rep in range(N_REPEATS):
            ss = 2000 + rep
            args = {"patient_name": case_ph1["patient"],
                    "true_schedule": case_ph1["true_schedule"],
                    "announced_schedule": case_ph1["announced_schedule"],
                    "sensor_seed": ss}
            df_b = run_day(**args)
            cfg = make_cfg(val)
            sup = Supervisor(clf=clf, feature_cols=feature_cols, cfg=cfg)
            df_s = run_day(**args, supervisor=sup)
            mb, ms = metrics(df_b), metrics(df_s)
            row = {param_name: val, "rep": rep}
            for k in mb:
                row[f"base_{k}"] = mb[k]
                row[f"sup_{k}"]  = ms[k]
            rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
# 1. alert threshold sweep
print("Threshold sweep:")
def cfg_th(th):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                            alert_th=th, cgm_gate=90, slope_gate=-0.50, n_consecutive=2)
df_th = run_sweep("alert_th", [0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.99], cfg_th)
display(df_th.groupby("alert_th")[["sup_minCGM", "sup_TAR", "sup_active_pct"]].mean().round(2))

In [ ]:
# 2. CGM gate sweep
print("CGM gate sweep:")
def cfg_cg(gate):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                            alert_th=0.97, cgm_gate=gate, slope_gate=-0.50, n_consecutive=2)
df_cg = run_sweep("cgm_gate", [80, 90, 100, 110], cfg_cg)
display(df_cg.groupby("cgm_gate")[["sup_minCGM", "sup_TAR", "sup_active_pct"]].mean().round(2))

In [ ]:
# 3. slope gate sweep (original — raw delta, NOT per-minute)
print("Slope gate sweep (original):")
def cfg_sg(sg):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                            alert_th=0.97, cgm_gate=90, slope_gate=sg, n_consecutive=2)
df_sg = run_sweep("slope_gate", [-0.25, -0.50, -1.0, -2.0], cfg_sg)
display(df_sg.groupby("slope_gate")[["sup_minCGM", "sup_TAR", "sup_active_pct"]].mean().round(2))
# NOTE: all values give identical results — the raw delta gate has no discriminative power
# this is one of the bugs that led to the corrected supervisor

In [ ]:
# 4. consecutive hits sweep
print("Consecutive hits sweep:")
def cfg_nc(nc):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                            alert_th=0.97, cgm_gate=90, slope_gate=-0.50, n_consecutive=nc)
df_nc = run_sweep("n_consecutive", [1, 2, 3], cfg_nc)
display(df_nc.groupby("n_consecutive")[["sup_minCGM", "sup_TAR", "sup_active_pct"]].mean().round(2))

In [ ]:
# 5. rescue carb dose sweep
print("Rescue dose sweep:")
def cfg_rc(dose):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=dose, basal_scale=0, bolus_scale=0,
                            alert_th=0.97, cgm_gate=90, slope_gate=-0.50, n_consecutive=2)
df_rc = run_sweep("rescue_g", [0, 4, 8, 10, 12, 16], cfg_rc)
display(df_rc.groupby("rescue_g")[["sup_minCGM", "sup_TAR", "sup_total_rescue_carbs"]].mean().round(2))

## Phase 2 — Corrected Supervisor

Two issues found during Phase 1:
1. **No training-domain guard** — model was queried below 80 mg/dL where it was never trained
2. **Slope gate had no effect** — raw `cgm_delta_15` is always < -0.25 during normal dynamics

Fixes applied in `PairedRiskSupervisor`:
- Hard-rule activation when CGM <= 80 (bypasses model)
- Slope converted to mg/dL per minute (delta_15 / 15)

In [ ]:
# switch to corrected supervisor for everything from here on
Supervisor = PairedRiskSupervisor

# verify the domain guard works
case_v = find_case(TEST_PATIENT, target="mild")
cfg_v = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                         alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)
sup_v = PairedRiskSupervisor(clf, feature_cols, cfg_v)
df_v = run_day(case_v["patient"], case_v["true_schedule"], case_v["announced_schedule"],
               sensor_seed=case_v["scenario_seed"], supervisor=sup_v)

print("Decision source breakdown:")
display(df_v["decision_source"].value_counts())
ml_rows = df_v[df_v["decision_source"] == "ml_model"]
if len(ml_rows) > 0:
    assert ml_rows["CGM"].min() > 80, "Model was called below 80!"
    print(f"ML model min CGM: {ml_rows['CGM'].min():.1f} — always above 80. OK.")

In [ ]:
# before vs after comparison
case_cmp = find_case(TEST_PATIENT, target="mild")
cfg_cmp = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=12, basal_scale=0, bolus_scale=0,
                           alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)
rows_cmp = []
for rep in range(N_REPEATS):
    ss = 5000 + rep
    args = {"patient_name": case_cmp["patient"],
            "true_schedule": case_cmp["true_schedule"],
            "announced_schedule": case_cmp["announced_schedule"],
            "sensor_seed": ss}
    df_b = run_day(**args)
    df_old = run_day(**args, supervisor=OriginalSupervisor(clf, feature_cols, cfg_cmp))
    df_new = run_day(**args, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_cmp))
    mb, mo, mn = metrics(df_b), metrics(df_old), metrics(df_new)
    rows_cmp.append({"rep": rep,
                     "base_minCGM": mb["minCGM"], "old_minCGM": mo["minCGM"], "new_minCGM": mn["minCGM"],
                     "base_TAR": mb["TAR"], "old_TAR": mo["TAR"], "new_TAR": mn["TAR"]})

df_cmp = pd.DataFrame(rows_cmp)
print("Before vs After (means):")
display(pd.DataFrame([
    {"Version": "Baseline",   "minCGM": df_cmp["base_minCGM"].mean(), "TAR": df_cmp["base_TAR"].mean()},
    {"Version": "Old (no guard)", "minCGM": df_cmp["old_minCGM"].mean(), "TAR": df_cmp["old_TAR"].mean()},
    {"Version": "Corrected",  "minCGM": df_cmp["new_minCGM"].mean(), "TAR": df_cmp["new_TAR"].mean()},
]).set_index("Version").round(2))

In [ ]:
# corrected slope gate sweep — should now show differentiation
print("Corrected slope gate sweep:")
case_sg2 = find_case(TEST_PATIENT, target="mild")
def cfg_sg2(sg):
    return SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=10, basal_scale=0, bolus_scale=0,
                            alert_th=0.95, cgm_gate=100, slope_gate=sg, n_consecutive=2)
rows_sg2 = []
for sg in [-0.25, -0.50, -0.75, -1.0, -1.5, -2.0]:
    print(f"  slope={sg} mg/dL/min")
    for rep in range(N_REPEATS):
        args = {"patient_name": case_sg2["patient"],
                "true_schedule": case_sg2["true_schedule"],
                "announced_schedule": case_sg2["announced_schedule"],
                "sensor_seed": 2000 + rep}
        df_b = run_day(**args)
        sup = PairedRiskSupervisor(clf, feature_cols, cfg_sg2(sg))
        df_s = run_day(**args, supervisor=sup)
        mb, ms = metrics(df_b), metrics(df_s)
        row = {"slope": sg, "rep": rep}
        for k in mb: row[f"base_{k}"] = mb[k]; row[f"sup_{k}"] = ms[k]
        rows_sg2.append(row)
df_sg2 = pd.DataFrame(rows_sg2)
display(df_sg2.groupby("slope")[["sup_minCGM", "sup_TAR", "sup_active_pct"]].mean().round(2))

## Three-Strategy Comparison

Using corrected supervisor with chosen operating point:
- **Attenuation**: suspend insulin only
- **Rescue carb**: 12g glucose, insulin unchanged
- **Hybrid**: suspend insulin + 12g rescue

In [ ]:
cfg_attn = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=0, basal_scale=0, bolus_scale=0,
                           alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)
cfg_rescue = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=12, basal_scale=1, bolus_scale=1,
                              alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)
cfg_hybrid = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=12, basal_scale=0, bolus_scale=0,
                              alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)

test_cases = [
    ("adult#002", "mild"), ("adult#002", "severe"),
    ("adult#008", "mild"), ("adult#008", "severe"),
]

all_strat = []
for pat, sev in test_cases:
    case = find_case(pat, target=sev)
    if case is None: continue
    for rep in range(N_REPEATS):
        args = {"patient_name": pat, "true_schedule": case["true_schedule"],
                "announced_schedule": case["announced_schedule"], "sensor_seed": 3000 + rep}
        df_b = run_day(**args)
        df_a = run_day(**args, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_attn))
        df_r = run_day(**args, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_rescue))
        df_h = run_day(**args, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_hybrid))
        mb, ma, mr, mh = metrics(df_b), metrics(df_a), metrics(df_r), metrics(df_h)
        row = {"case": f"{pat}_{sev}", "rep": rep}
        for pfx, m in [("base", mb), ("attn", ma), ("rescue", mr), ("hybrid", mh)]:
            for k, v in m.items(): row[f"{pfx}_{k}"] = v
        all_strat.append(row)

df_strat = pd.DataFrame(all_strat)
df_strat.to_csv(CSVS / "strategy_comparison.csv", index=False)

# summary per case
for case_lbl in df_strat["case"].unique():
    sub = df_strat[df_strat["case"] == case_lbl]
    print(f"\n--- {case_lbl} ---")
    for pfx, name in [("base","Baseline"), ("attn","Attenuation"), ("rescue","Rescue"), ("hybrid","Hybrid")]:
        print(f"  {name:15s} minCGM={sub[f'{pfx}_minCGM'].mean():.1f}  "
              f"TBR70={sub[f'{pfx}_TBR70'].mean():.2f}  TAR={sub[f'{pfx}_TAR'].mean():.1f}")

## Full 10-Patient Validation

One hypo-prone scenario per patient (baseline minCGM between 40–70),
N=20 sensor-noise repeats with the hybrid supervisor.

In [ ]:
ALL_PATIENTS = [f"adult#{i:03d}" for i in range(1, 11)]

VAL_CASES = []
for pat in ALL_PATIENTS:
    pnum = int(pat.split("#")[1])
    c = find_case_range(pat, 40, 70, seed_start=SEED_START + pnum * MAX_TRIES)
    if c: VAL_CASES.append(c)
print(f"\nFound {len(VAL_CASES)} validation cases")

cfg_val = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=12, basal_scale=0, bolus_scale=0,
                           alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)

In [ ]:
val_rows = []
for case in VAL_CASES:
    pat = case["patient"]
    print(f"\n{pat} (seed={case['scenario_seed']})")
    for rep in range(N_REPEATS):
        args = {"patient_name": pat, "true_schedule": case["true_schedule"],
                "announced_schedule": case["announced_schedule"], "sensor_seed": 4000 + rep}
        df_b = run_day(**args)
        sup = PairedRiskSupervisor(clf, feature_cols, cfg_val)
        df_h = run_day(**args, supervisor=sup)
        mb, mh = metrics(df_b), metrics(df_h)
        row = {"patient": pat, "rep": rep}
        for k in mb: row[f"base_{k}"] = mb[k]; row[f"hybrid_{k}"] = mh[k]
        row["delta_minCGM"] = mh["minCGM"] - mb["minCGM"]
        val_rows.append(row)

df_val = pd.DataFrame(val_rows)
df_val.to_csv(CSVS / "validation_results.csv", index=False)

# headline
print(f"\n=== HEADLINE ===")
print(f"Baseline mean minCGM: {df_val['base_minCGM'].mean():.1f}")
print(f"Hybrid mean minCGM:   {df_val['hybrid_minCGM'].mean():.1f}")
print(f"Mean improvement:     {df_val['delta_minCGM'].mean():.1f} mg/dL")

## Final 3-Config Comparison

- **Config A**: Hybrid balanced — 12g, threshold 0.95
- **Config B**: Hybrid aggressive — 16g, threshold 0.80, n_consecutive=1
- **Config C**: Rescue-only — 12g, threshold 0.95, insulin unchanged

In [ ]:
cfg_A = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=12, basal_scale=0, bolus_scale=0,
                        alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)
cfg_B = SupervisorConfig(mode=SupervisorMode.HYBRID, rescue_carb_g=16, basal_scale=0, bolus_scale=0,
                         alert_th=0.80, cgm_gate=100, slope_gate=-0.25, n_consecutive=1)
cfg_C = SupervisorConfig(mode=SupervisorMode.RESCUE_CARB, rescue_carb_g=12, basal_scale=1, bolus_scale=1,
                         alert_th=0.95, cgm_gate=100, slope_gate=-0.50, n_consecutive=2)

In [ ]:
# find cases (same approach as validation)
VAL_FINAL = []
for pat in ALL_PATIENTS:
    pnum = int(pat.split("#")[1])
    c = find_case_range(pat, 40, 70, seed_start=SEED_START + pnum * MAX_TRIES)
    if c: VAL_FINAL.append(c)
print(f"Found {len(VAL_FINAL)} cases")

In [ ]:
# main loop
rows_f3 = []
for case in VAL_FINAL:
    pat = case["patient"]
    print(f"\n{pat}")
    for rep in range(N_REPEATS):
        args = {"patient_name": pat, "true_schedule": case["true_schedule"],
                "announced_schedule": case["announced_schedule"], "sensor_seed": 7000 + rep}
        df_b = run_day(**args)
        mb = metrics(df_b)
        row = {"patient": pat, "rep": rep}
        for k in mb: row[f"base_{k}"] = mb[k]
        for lbl, cfg in [("A", cfg_A), ("B", cfg_B), ("C", cfg_C)]:
            sup = PairedRiskSupervisor(clf, feature_cols, cfg)
            df_s = run_day(**args, supervisor=sup)
            ms = metrics(df_s)
            for k in ms: row[f"{lbl}_{k}"] = ms[k]
        rows_f3.append(row)

df_f3 = pd.DataFrame(rows_f3)
df_f3.to_csv(CSVS / "final3_results.csv", index=False)
print(f"\n{len(df_f3)} rows saved")

In [ ]:
# headline table
headline = pd.DataFrame([
    {"Config": "Baseline",
     "minCGM": df_f3["base_minCGM"].mean(), "TBR70": df_f3["base_TBR70"].mean(),
     "TBR54": df_f3["base_TBR54"].mean(), "TIR": df_f3["base_TIR"].mean(),
     "TAR": df_f3["base_TAR"].mean(), "Rescue (g)": df_f3["base_total_rescue_carbs"].mean()},
    {"Config": "A: Hybrid balanced",
     "minCGM": df_f3["A_minCGM"].mean(), "TBR70": df_f3["A_TBR70"].mean(),
     "TBR54": df_f3["A_TBR54"].mean(), "TIR": df_f3["A_TIR"].mean(),
     "TAR": df_f3["A_TAR"].mean(), "Rescue (g)": df_f3["A_total_rescue_carbs"].mean()},
    {"Config": "B: Hybrid aggressive",
     "minCGM": df_f3["B_minCGM"].mean(), "TBR70": df_f3["B_TBR70"].mean(),
     "TBR54": df_f3["B_TBR54"].mean(), "TIR": df_f3["B_TIR"].mean(),
     "TAR": df_f3["B_TAR"].mean(), "Rescue (g)": df_f3["B_total_rescue_carbs"].mean()},
    {"Config": "C: Rescue-only",
     "minCGM": df_f3["C_minCGM"].mean(), "TBR70": df_f3["C_TBR70"].mean(),
     "TBR54": df_f3["C_TBR54"].mean(), "TIR": df_f3["C_TIR"].mean(),
     "TAR": df_f3["C_TAR"].mean(), "Rescue (g)": df_f3["C_total_rescue_carbs"].mean()},
]).set_index("Config")
print("\n" + "=" * 60)
print("HEADLINE — 3-CONFIG COMPARISON")
print("=" * 60)
display(headline.round(2))
headline.to_csv(CSVS / "final3_headline.csv")

In [ ]:
# per-patient wilcoxon tests
stats = []
for pat, g in df_f3.groupby("patient"):
    row = {"patient": pat}
    for lbl in ["A", "B", "C"]:
        row[f"{lbl}_p_minCGM"] = safe_wilcoxon(g[f"{lbl}_minCGM"], g["base_minCGM"], "greater")
        row[f"{lbl}_p_TBR70"]  = safe_wilcoxon(g[f"{lbl}_TBR70"],  g["base_TBR70"],  "less")
    stats.append(row)
df_stats = pd.DataFrame(stats).sort_values("patient")
display(df_stats.round(4))
df_stats.to_csv(CSVS / "final3_stats.csv", index=False)

## Report figures

In [ ]:
# 2x2 bar chart — nadir, TBR70, TAR, rescue cost
configs = ["Baseline", "A: Hybrid\nbalanced", "B: Hybrid\naggressive", "C: Rescue\nonly"]
x = np.arange(4)
cols = ["tab:blue", "tab:orange", "tab:red", "tab:green"]

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
for ax, metric, ylabel, title, fmt in [
    (axes[0,0], "minCGM", "Mean minCGM (mg/dL)", "(a) Nadir Glucose (higher = safer)", ".1f"),
    (axes[0,1], "TBR70",  "Mean TBR70 (%)",       "(b) Time Below 70 (lower = better)", ".2f"),
    (axes[1,0], "TAR",    "Mean TAR (%)",          "(c) Time Above 180 (lower = better)", ".1f"),
    (axes[1,1], "total_rescue_carbs", "Mean rescue (g)", "(d) Intervention Cost", ".1f"),
]:
    vals = [df_f3[f"{p}_{metric}"].mean() for p in ["base", "A", "B", "C"]]
    bars = ax.bar(x, vals, color=cols, edgecolor="black", lw=0.3)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12)
    for b, v in zip(bars, vals):
        ax.text(b.get_x() + b.get_width()/2, v + 0.3, f"{v:{fmt}}", ha="center", fontsize=10)
    ax.set_xticks(x)
    ax.set_xticklabels(configs, fontsize=10)
    if metric == "minCGM":
        ax.axhline(70, color="gray", ls="--", lw=1)
fig.suptitle(f"Final Comparison: 3 Configs (10 patients, N={N_REPEATS})", fontsize=14, y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(FIGS / "fig_final3_comparison_2x2.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# per-patient nadir bar chart
pp = df_f3.groupby("patient").mean(numeric_only=True).sort_index()
patients = pp.index
x = np.arange(len(patients))
w = 0.20

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - 1.5*w, pp["base_minCGM"], w, label="Baseline", color="tab:blue", edgecolor="black", lw=0.3)
ax.bar(x - 0.5*w, pp["A_minCGM"], w, label="A: Hybrid balanced", color="tab:orange", edgecolor="black", lw=0.3)
ax.bar(x + 0.5*w, pp["B_minCGM"], w, label="B: Hybrid aggressive", color="tab:red", edgecolor="black", lw=0.3)
ax.bar(x + 1.5*w, pp["C_minCGM"], w, label="C: Rescue-only", color="tab:green", edgecolor="black", lw=0.3)
ax.axhline(70, color="gray", ls="--")
ax.axhline(54, color="red", ls="--", alpha=0.5)
ax.set_ylabel("Mean min CGM (mg/dL)", fontsize=12)
ax.set_xlabel("Patient", fontsize=12)
ax.set_title(f"Per-Patient Nadir: Baseline vs 3 Configs (N={N_REPEATS})", fontsize=13)
ax.set_xticks(x)
ax.set_xticklabels(patients, rotation=45, ha="right", fontsize=10)
ax.legend(loc="lower right", fontsize=10)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS / "fig_final3_perpatient_minCGM.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# representative matched trace — pick patient with deepest baseline hypo
best_case = min(VAL_FINAL, key=lambda c: c["baseline_minCGM"])
TRACE_PAT = best_case["patient"]
print(f"Trace patient: {TRACE_PAT} (baseline minCGM={best_case['baseline_minCGM']:.1f})")

common = {"patient_name": TRACE_PAT, "true_schedule": best_case["true_schedule"],
          "announced_schedule": best_case["announced_schedule"], "sensor_seed": 7000}
df_tb = run_day(**common)
df_tA = run_day(**common, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_A))
df_tB = run_day(**common, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_B))
df_tC = run_day(**common, supervisor=PairedRiskSupervisor(clf, feature_cols, cfg_C))

print(f"minCGM: base={df_tb['CGM'].min():.1f}  A={df_tA['CGM'].min():.1f}  "
      f"B={df_tB['CGM'].min():.1f}  C={df_tC['CGM'].min():.1f}")

# two-panel: CGM traces + p(hypo)
fig = plt.figure(figsize=(13, 8))
gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.15)
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1], sharex=ax1)

for df, lbl, col, lw in [(df_tb, "Baseline", "tab:blue", 2),
                           (df_tA, "A: Hybrid balanced", "tab:orange", 1.4),
                           (df_tB, "B: Hybrid aggressive", "tab:red", 1.4),
                           (df_tC, "C: Rescue-only", "tab:green", 1.4)]:
    ax1.plot(df["time"], df["CGM"], color=col, lw=lw, label=lbl)
ax1.axhline(70, ls="--", lw=1, color="gray", alpha=0.8)
ax1.axhline(54, ls="--", lw=0.8, color="red", alpha=0.4)

# mark first activation per config
for df, col in [(df_tA, "tab:orange"), (df_tB, "tab:red"), (df_tC, "tab:green")]:
    act = df["supervisor_active"].values
    for i in range(1, len(act)):
        if act[i] == 1 and act[i-1] == 0:
            ax1.plot(df["time"].iloc[i], df["CGM"].iloc[i],
                     marker="v", ms=9, color=col, markeredgecolor="black", mew=0.6, zorder=6)
            break
shade_active(ax1, df_tA["time"], df_tA["supervisor_active"])

ax1.set_ylabel("CGM (mg/dL)", fontsize=12)
ax1.legend(loc="upper right", fontsize=10)
ax1.set_title(f"Representative Trace — {TRACE_PAT}", fontsize=13)
ax1.grid(True, ls="--", lw=0.4, alpha=0.3)
plt.setp(ax1.get_xticklabels(), visible=False)

# p(hypo) panel
pv = df_tA["p_hypo"].copy()
ax2.fill_between(df_tA["time"], pv.fillna(0), alpha=0.25, color="tab:orange")
ax2.plot(df_tA["time"], pv, color="tab:orange", lw=1, label="p(hypo)")
ax2.axhline(0.95, ls="--", color="gray", lw=0.8, label=r"$\theta$=0.95 (A,C)")
ax2.axhline(0.80, ls="--", color="tab:red", lw=0.8, alpha=0.6, label=r"$\theta$=0.80 (B)")
ax2.set_ylabel("p(hypo)", fontsize=12)
ax2.set_xlabel("Time", fontsize=12)
ax2.set_ylim(-0.02, 1.05)
ax2.legend(loc="upper right", fontsize=9, ncol=3)
ax2.grid(True, ls="--", lw=0.4, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
fig.savefig(FIGS / "fig_final3_matched_trace.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# print all numbers for latex tables
print("=" * 70)
print("NUMBERS FOR REPORT TABLES")
print("=" * 70)

for pfx, name in [("base","Baseline"), ("A","Config A"), ("B","Config B"), ("C","Config C")]:
    mc  = df_f3[f"{pfx}_minCGM"].mean()
    t70 = df_f3[f"{pfx}_TBR70"].mean()
    t54 = df_f3[f"{pfx}_TBR54"].mean()
    tir = df_f3[f"{pfx}_TIR"].mean()
    tar = df_f3[f"{pfx}_TAR"].mean()
    rc  = df_f3[f"{pfx}_total_rescue_carbs"].mean()
    print(f"  {name:20s} minCGM={mc:.2f}  TBR70={t70:.2f}  TBR54={t54:.2f}  "
          f"TIR={tir:.2f}  TAR={tar:.2f}  rescue={rc:.1f}g")

print("\nPer-patient nadirs:")
for pat in sorted(df_f3["patient"].unique()):
    g = df_f3[df_f3["patient"] == pat]
    print(f"  {pat}  base={g['base_minCGM'].mean():.1f}  A={g['A_minCGM'].mean():.1f}  "
          f"B={g['B_minCGM'].mean():.1f}  C={g['C_minCGM'].mean():.1f}")